In [1]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor

from core import (
    GeoMEGBenchmark, BenchmarkConfig, regression_metrics,
    field_direction, deadzone_laser, sarvas_field_vector, local_field_artifacts,
    choose_subset_indices, unit
)

In [2]:
ROOT = Path('C:\\Users\\scx20\\Downloads\\Physics_conditioned_residual_learning_for_source_localization_in_unshielded_scalar_OPM_MEG__a_spherical_conductor_benchmark_with_continuous_off_grid_s')

In [3]:
OUT = ROOT / 'outputs'
TABLES = OUT / 'tables'
DATA = OUT / 'data'
for p in [TABLES, DATA]:
    p.mkdir(parents=True, exist_ok=True)


def fit_regressor(X: np.ndarray, Y: np.ndarray, *, n_estimators: int = 180, min_samples_leaf: int = 2, random_state: int = 0):
    reg = ExtraTreesRegressor(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state,
        n_jobs=1,
    )
    reg.fit(X, Y)
    return reg


def pred_model(name: str, split: dict, models: dict, masks: dict):
    if name == 'geomeg':
        return models[name].predict(split['full'])
    if name == 'raw':
        return models[name].predict(split['raw'])
    if name == 'no_cent':
        return models[name].predict(split['no_cent'])
    if name == 'cent_only':
        return models[name].predict(split['cent_only'])
    if name == 'no_nuis':
        return models[name].predict(split['full'][:, masks['no_nuis']])
    if name == 'scores_nuis':
        return models[name].predict(split['full'][:, masks['scores_nuis']])
    if name == 'coords_cent':
        return models[name].predict(split['full'][:, masks['coords_cent']])
    if name == 'nuis_only':
        return models[name].predict(split['full'][:, masks['nuis_only']])
    if name == 'grid':
        return split['grid']
    if name == 'mne':
        return split['mne']
    if name == 'centroid':
        return split['centroid']
    raise KeyError(name)


def detailed_df(split_name: str, split: dict, pred_names: list[str], models: dict, masks: dict) -> pd.DataFrame:
    truth = split['truth']
    metas = split['meta']
    preds_cache = {m: pred_model(m, split, models, masks) for m in pred_names}
    rows = []
    for i in range(len(truth)):
        row = {
            'split': split_name, 'sample_id': i,
            'true_x_m': truth[i, 0], 'true_y_m': truth[i, 1], 'true_z_m': truth[i, 2]
        }
        row.update(metas[i].copy())
        for m, arr in preds_cache.items():
            row[f'{m}_x_m'] = arr[i, 0]
            row[f'{m}_y_m'] = arr[i, 1]
            row[f'{m}_z_m'] = arr[i, 2]
            row[f'{m}_err_mm'] = float(np.linalg.norm(arr[i] - truth[i]) * 1000.0)
        rows.append(row)
    return pd.DataFrame(rows)


def metrics_for(dfsub: pd.DataFrame, methods: list[str]) -> pd.DataFrame:
    rows = []
    for m in methods:
        err = dfsub[f'{m}_err_mm'].to_numpy()
        rows.append({'method': m, **regression_metrics(err), 'n': len(dfsub)})
    return pd.DataFrame(rows)


def polar_bin(p):
    if p < 25:
        return '0-25'
    if p < 45:
        return '25-45'
    if p < 70:
        return '45-70'
    if p < 85:
        return '75-85'
    if p < 95:
        return '85-95'
    return '95-105'


def depth_bin(r):
    mm = 1000 * r
    if mm < 66.7:
        return '60-66.7'
    if mm < 73.3:
        return '66.7-73.3'
    return '73.3-80'


def eta_squared(series: pd.Series, groups: pd.Series) -> float:
    y = series.to_numpy()
    grand = y.mean()
    ss_total = ((y - grand) ** 2).sum()
    ss_between = 0.0
    tmp = pd.DataFrame({'y': y, 'g': groups})
    for _, sub in tmp.groupby('g', observed=False):
        mu = sub['y'].mean()
        ss_between += len(sub) * (mu - grand) ** 2
    return float(ss_between / ss_total) if ss_total > 0 else 0.0


def group_results(df: pd.DataFrame, factor: str, methods: list[str]) -> pd.DataFrame:
    rows = []
    for split in ['IID', 'OOD']:
        sub = df[df['split'] == split]
        for level, ss in sub.groupby(factor, observed=False):
            row = {'split': split, 'factor': factor, 'level': str(level), 'n': len(ss)}
            for m in methods:
                row[f'{m}_mean_mm'] = ss[f'{m}_err_mm'].mean()
            rows.append(row)
    return pd.DataFrame(rows)


def simulate_stress_sample(bench: GeoMEGBenchmark, rng: np.random.Generator, split='ood', sensor_jitter_mm=0.0, second_source_scale=0.0):
    pos, q = bench.sample_continuous_source(rng)
    if split == 'ood':
        polar_deg = rng.uniform(75, 105)
        grad_scale = rng.uniform(5, 10)
        ext_white = rng.uniform(4, 9)
        count = int(rng.choice(bench.cfg.ood_counts))
    else:
        polar_deg = rng.uniform(0, 70)
        grad_scale = rng.uniform(0, 6)
        ext_white = rng.uniform(0, 6)
        count = int(rng.choice(bench.cfg.train_counts))
    azim_deg = rng.uniform(0, 360)
    n_field = field_direction(polar_deg, azim_deg)
    intrinsic = float(rng.choice(bench.cfg.intrinsic_levels))
    dead_mode = str(rng.choice(bench.cfg.dead_modes))
    active = choose_subset_indices(count, bench.cfg.n_sensors)

    sensors_true = bench.sensors.copy()
    if sensor_jitter_mm > 0:
        jitter = rng.normal(0, sensor_jitter_mm / 1000.0, size=sensors_true.shape)
        sensors_true = sensors_true + jitter
        sensors_true = np.array([
            unit(np.array([p[0], p[1], abs(p[2])])) * bench.cfg.sensor_r for p in sensors_true
        ])

    Bvec = sarvas_field_vector(pos, q, sensors_true) * 1e15
    if second_source_scale > 0:
        for _ in range(50):
            pos2, q2 = bench.sample_continuous_source(rng)
            if np.linalg.norm(pos2 - pos) > 0.03:
                break
        q2 = q2 * float(second_source_scale)
        Bvec += sarvas_field_vector(pos2, q2, sensors_true) * 1e15

    signal = Bvec @ n_field
    y = np.zeros(bench.cfg.n_sensors, dtype=np.float32)
    mask = np.zeros(bench.cfg.n_sensors, dtype=np.float32)
    sin2_vals = []
    for i in active:
        mask[i] = 1
        laser = deadzone_laser(sensors_true[i], n_field, dead_mode, rng)
        s = float(np.clip(1 - float(np.dot(laser, n_field) ** 2), 0.05, 1.0))
        sin2_vals.append(s)
        sigma = ((intrinsic ** 2) / s + ext_white ** 2) ** 0.5
        gain = 1 - 0.18 * (1 - s) + rng.normal(0, 0.02)
        artifact = local_field_artifacts(sensors_true[i], bench.cfg.sensor_r, rng, grad_scale, azim_deg)
        y[i] = gain * signal[i] + artifact + rng.normal(0, sigma)

    meta = {
        'count': count, 'intrinsic': intrinsic, 'ext_white': float(ext_white),
        'grad_scale': float(grad_scale), 'polar_deg': float(polar_deg),
        'azim_deg': float(azim_deg), 'dead_mode': dead_mode,
        'sin2mean': float(np.mean(sin2_vals)), 'nField': n_field,
        'source_r': float(np.linalg.norm(pos)),
        'nField_x': float(n_field[0]), 'nField_y': float(n_field[1]), 'nField_z': float(n_field[2]),
        'sensor_jitter_mm': float(sensor_jitter_mm), 'second_source_scale': float(second_source_scale),
    }
    feats = bench.feature_pack(y, mask, meta)
    return pos.astype(np.float32), meta, y, mask, feats


def build_stress_dataset(bench: GeoMEGBenchmark, n=300, split='ood', sensor_jitter_mm=0.0, second_source_scale=0.0, seed=0):
    rng = np.random.default_rng(seed)
    out = {'truth': [], 'meta': [], 'y': [], 'mask': [], 'full': [], 'raw': [], 'no_cent': [], 'cent_only': [], 'grid': [], 'mne': [], 'centroid': []}
    for _ in range(n):
        pos, meta, y, mask, feats = simulate_stress_sample(bench, rng, split=split, sensor_jitter_mm=sensor_jitter_mm, second_source_scale=second_source_scale)
        out['truth'].append(pos); out['meta'].append(meta); out['y'].append(y); out['mask'].append(mask)
        for k in ['full', 'raw', 'no_cent', 'cent_only', 'grid', 'mne', 'centroid']:
            out[k].append(feats[k])
    for k, v in out.items():
        if k != 'meta':
            out[k] = np.stack(v)
    return out

In [4]:
bench = GeoMEGBenchmark(BenchmarkConfig())
train = bench.build_dataset(4000, 'train', seed=11)
iid = bench.build_dataset(1000, 'train', seed=12)
ood = bench.build_dataset(1000, 'ood', seed=13)

feature_names = np.array(bench.feature_names)
coords_mask = np.array([name.endswith(('_x', '_y', '_z')) and name.startswith('top') for name in feature_names])
stats_mask = np.array([name.startswith('top') and not name.endswith(('_x', '_y', '_z')) for name in feature_names])
cent_mask = np.array([name.startswith('cent_') for name in feature_names])
nuis_mask = ~(coords_mask | stats_mask | cent_mask)
masks = {
    'no_nuis': ~nuis_mask,
    'scores_nuis': stats_mask | nuis_mask | cent_mask,
    'coords_cent': coords_mask | cent_mask,
    'nuis_only': nuis_mask,
    'coords_mask': coords_mask,
    'stats_mask': stats_mask,
    'cent_mask': cent_mask,
    'nuis_mask': nuis_mask,
}

models = {
    'geomeg': fit_regressor(train['full'], train['truth'], random_state=0),
    'raw': fit_regressor(train['raw'], train['truth'], random_state=0),
    'no_cent': fit_regressor(train['no_cent'], train['truth'], random_state=0),
    'cent_only': fit_regressor(train['cent_only'], train['truth'], random_state=0),
    'no_nuis': fit_regressor(train['full'][:, masks['no_nuis']], train['truth'], random_state=0),
    'scores_nuis': fit_regressor(train['full'][:, masks['scores_nuis']], train['truth'], random_state=0),
    'coords_cent': fit_regressor(train['full'][:, masks['coords_cent']], train['truth'], random_state=0),
    'nuis_only': fit_regressor(train['full'][:, masks['nuis_only']], train['truth'], random_state=0),
}

methods = ['geomeg', 'centroid', 'grid', 'mne', 'raw']
ablations = ['geomeg', 'no_cent', 'cent_only', 'no_nuis', 'scores_nuis', 'coords_cent', 'nuis_only', 'raw']

pred_names = methods + [m for m in ablations if m not in methods]
df_iid = detailed_df('IID', iid, pred_names, models, masks)
df_ood = detailed_df('OOD', ood, pred_names, models, masks)
df = pd.concat([df_iid, df_ood], ignore_index=True)

df['polar_bin'] = df['polar_deg'].map(polar_bin)
df['depth_bin'] = df['source_r'].map(depth_bin)
df['grad_bin'] = pd.qcut(df['grad_scale'], q=3, labels=['low', 'mid', 'high'], duplicates='drop')
df['ext_bin'] = pd.qcut(df['ext_white'], q=3, labels=['low', 'mid', 'high'], duplicates='drop')
df['sin2_bin'] = pd.Series(np.where(df['sin2mean'] < 0.4, 'poor', np.where(df['sin2mean'] < 0.85, 'mid', 'good')))
df['count_bin'] = df['count'].astype(int).astype(str)

summary_rows = []
for split_name in ['IID', 'OOD']:
    sub = df[df['split'] == split_name]
    tmp = metrics_for(sub, methods)
    tmp.insert(0, 'split', split_name)
    summary_rows.append(tmp)
summary = pd.concat(summary_rows, ignore_index=True)
summary.to_csv(TABLES / 'summary_results.csv', index=False)

abl_rows = []
for split_name in ['IID', 'OOD']:
    sub = df[df['split'] == split_name]
    tmp = metrics_for(sub, ablations)
    tmp.insert(0, 'split', split_name)
    abl_rows.append(tmp)
pd.concat(abl_rows, ignore_index=True).to_csv(TABLES / 'ablation_results.csv', index=False)

factors = {
    'channel_count': 'count_bin',
    'intrinsic_noise': 'intrinsic',
    'external_noise': 'ext_bin',
    'residual_field': 'grad_bin',
    'field_orientation': 'polar_bin',
    'dead_zone': 'dead_mode',
    'source_radius': 'depth_bin',
    'domain_shift': 'split',
}
eta_rows = []
for method in methods:
    for fac, col in factors.items():
        eta_rows.append({'method': method, 'factor': fac, 'eta_sq': eta_squared(df[f'{method}_err_mm'], df[col])})
pd.DataFrame(eta_rows).to_csv(TABLES / 'factor_eta_squared.csv', index=False)

group_imp = pd.DataFrame({
    'group': ['top_score_stats', 'top_candidate_coords', 'centroid_coords', 'nuisance'],
    'importance': [
        models['geomeg'].feature_importances_[stats_mask].sum(),
        models['geomeg'].feature_importances_[coords_mask].sum(),
        models['geomeg'].feature_importances_[cent_mask].sum(),
        models['geomeg'].feature_importances_[nuis_mask].sum(),
    ]
}).sort_values('importance', ascending=False)
group_imp['fraction'] = group_imp['importance'] / group_imp['importance'].sum()
group_imp.to_csv(TABLES / 'feature_group_importance.csv', index=False)

for factor in ['count_bin', 'dead_mode', 'grad_bin', 'ext_bin', 'polar_bin', 'depth_bin']:
    group_results(df, factor, methods).to_csv(TABLES / f'{factor}_group_results.csv', index=False)

# cluster-geometry analysis
preds_geo_iid = pred_model('geomeg', iid, models, masks)
preds_geo_ood = pred_model('geomeg', ood, models, masks)
rows = []
for split_name, split, pred_geo in [('IID', iid, preds_geo_iid), ('OOD', ood, preds_geo_ood)]:
    cent_err = np.linalg.norm(split['centroid'] - split['truth'], axis=1) * 1000.0
    geo_err = np.linalg.norm(pred_geo - split['truth'], axis=1) * 1000.0
    for i in range(len(split['truth'])):
        top = split['top_idx'][i]
        cand = bench.candidates[top]
        sc = split['score_map'][i][top]
        w = np.exp(sc - sc.max())
        w = w / w.sum()
        cent = (cand * w[:, None]).sum(axis=0)
        cov = ((cand - cent).T * w) @ (cand - cent)
        eigvals = np.sort(np.linalg.eigvalsh(cov))[::-1]
        anis = float(eigvals[0] / max(eigvals[1], 1e-12))
        spread = float(np.sqrt(eigvals.sum()) * 1000.0)
        rows.append({
            'split': split_name, 'sample_id': i, 'anisotropy': anis, 'spread_mm': spread,
            'centroid_err_mm': float(cent_err[i]), 'geomeg_err_mm': float(geo_err[i]),
            'gain_mm': float(cent_err[i] - geo_err[i]),
        })
cg = pd.DataFrame(rows)
cg['spread_bin'] = pd.qcut(cg['spread_mm'], q=3, labels=['compact', 'medium', 'diffuse'])
cg.to_csv(TABLES / 'cluster_geometry_analysis.csv', index=False)

# stress tests
stress_sets = {
    'OOD_nominal': ood,
    'OOD_sensor_jitter_2mm': build_stress_dataset(bench, n=300, split='ood', sensor_jitter_mm=2.0, seed=21),
    'OOD_sensor_jitter_4mm': build_stress_dataset(bench, n=300, split='ood', sensor_jitter_mm=4.0, seed=22),
    'OOD_two_source_0p3': build_stress_dataset(bench, n=300, split='ood', second_source_scale=0.3, seed=23),
    'OOD_two_source_0p6': build_stress_dataset(bench, n=300, split='ood', second_source_scale=0.6, seed=24),
}
stress_rows = []
for sname, split in stress_sets.items():
    for method in methods:
        pred = pred_model(method, split, models, masks)
        err = np.linalg.norm(pred - split['truth'], axis=1) * 1000.0
        stress_rows.append({'scenario': sname, 'method': method, **regression_metrics(err), 'n': len(err)})
pd.DataFrame(stress_rows).to_csv(TABLES / 'stress_test_results.csv', index=False)

# save per-sample detailed predictions
df_iid.to_csv(TABLES / 'detailed_iid_predictions.csv', index=False)
df_ood.to_csv(TABLES / 'detailed_ood_predictions.csv', index=False)

# save arrays and representative indices for figure generation
iid_gain = np.linalg.norm(iid['centroid'] - iid['truth'], axis=1) * 1000.0 - np.linalg.norm(preds_geo_iid - iid['truth'], axis=1) * 1000.0
ood_gain = np.linalg.norm(ood['centroid'] - ood['truth'], axis=1) * 1000.0 - np.linalg.norm(preds_geo_ood - ood['truth'], axis=1) * 1000.0
rep_iid = int(np.argsort(iid_gain)[-5])
rep_ood = int(np.argmax(ood_gain))


In [7]:
np.savez_compressed(
    DATA / 'benchmark_core.npz',
    sensors=bench.sensors, candidates=bench.candidates,
    iid_truth=iid['truth'], iid_score=iid['score_map'], iid_top_idx=iid['top_idx'], iid_y=iid['y'], iid_mask=iid['mask'],
    ood_truth=ood['truth'], ood_score=ood['score_map'], ood_top_idx=ood['top_idx'], ood_y=ood['y'], ood_mask=ood['mask'],
    iid_centroid=iid['centroid'], iid_grid=iid['grid'], iid_mne=iid['mne'], iid_geomeg=preds_geo_iid,
    ood_centroid=ood['centroid'], ood_grid=ood['grid'], ood_mne=ood['mne'], ood_geomeg=preds_geo_ood,
    rep_iid=np.array([rep_iid]), rep_ood=np.array([rep_ood])
)

In [8]:
arr = np.load(DATA / 'benchmark_core.npz', allow_pickle=True)
print(arr.files)

['sensors', 'candidates', 'iid_truth', 'iid_score', 'iid_top_idx', 'iid_y', 'iid_mask', 'ood_truth', 'ood_score', 'ood_top_idx', 'ood_y', 'ood_mask', 'iid_centroid', 'iid_grid', 'iid_mne', 'iid_geomeg', 'ood_centroid', 'ood_grid', 'ood_mne', 'ood_geomeg', 'rep_iid', 'rep_ood']


In [9]:
rep_ood

802

In [10]:
rep_iid

712